In [21]:
import pandas as pd
import numpy as np
import os

# Load processed signal data (daily raw + weekday raw)
DATA_FOLDER = "Signal_output_split" 
daily_raw_path = os.path.join(DATA_FOLDER, "signal_daily_raw.csv")
weekday_raw_path = os.path.join(DATA_FOLDER, "signal_weekday_avg_raw.csv")

daily_df = pd.read_csv(daily_raw_path)
weekday_df = pd.read_csv(weekday_raw_path)
print("Daily shape:", daily_df.shape)
print("Weekday shape:", weekday_df.shape)

Daily shape: (49454, 35)
Weekday shape: (1890, 36)


In [22]:
# Compute turning ratios for each signal using all directions
def compute_overall_turn_ratios(df):
    df = df.copy()

    # All L/T/R columns after TL/TR splitting
    left_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_L")]
    through_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_T")]
    right_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_R")]

    # Sum across all approaches for each row
    df["left_total"] = df[left_cols].sum(axis=1)
    df["through_total"] = df[through_cols].sum(axis=1)
    df["right_total"] = df[right_cols].sum(axis=1)

    df["turn_total"] = df["left_total"] + df["through_total"] + df["right_total"]

    # Compute ratios
    df["left_ratio"] = np.where(df["turn_total"] > 0,
                                df["left_total"] / df["turn_total"], np.nan)
    df["through_ratio"] = np.where(df["turn_total"] > 0,
                                   df["through_total"] / df["turn_total"], np.nan)
    df["right_ratio"] = np.where(df["turn_total"] > 0,
                                 df["right_total"] / df["turn_total"], np.nan)

    return df


daily_ratio = compute_overall_turn_ratios(daily_df)
weekday_ratio = compute_overall_turn_ratios(weekday_df)

# Keep only rows with actual turning volume
daily_ratio = daily_ratio[daily_ratio["turn_total"] > 0].copy()
weekday_ratio = weekday_ratio[weekday_ratio["turn_total"] > 0].copy()

print("Remaining signals in daily_ratio:", daily_ratio["SignalID"].nunique())
print("Remaining signals in weekday_ratio:", weekday_ratio["SignalID"].nunique())


Remaining signals in daily_ratio: 264
Remaining signals in weekday_ratio: 264


In [23]:
# Station-level overall L/T/R ratios
station_totals = (
    daily_ratio.groupby("SignalID", as_index=False)[
        ["left_total", "through_total", "right_total"]
    ].sum()
)

station_totals["turn_total"] = (
    station_totals["left_total"] +
    station_totals["through_total"] +
    station_totals["right_total"]
)

station_totals["left_ratio"] = station_totals["left_total"] / station_totals["turn_total"]
station_totals["through_ratio"] = station_totals["through_total"] / station_totals["turn_total"]
station_totals["right_ratio"] = station_totals["right_total"] / station_totals["turn_total"]

print("\nStation-level L/T/R ratios:")
print(station_totals[["SignalID", "left_ratio", "through_ratio", "right_ratio"]].head())

# Save for inspection
station_ratio_path = os.path.join(DATA_FOLDER, "signal_station_ratio_check_split.csv")
station_totals.to_csv(station_ratio_path, index=False)
print("Saved station-level ratio check to:", station_ratio_path)



Station-level L/T/R ratios:
   SignalID  left_ratio  through_ratio  right_ratio
0        75    0.467383       0.327933     0.204684
1       161    0.375000       0.500000     0.125000
2       178    0.145429       0.554579     0.299992
3       195    0.026390       0.891339     0.082271
4       196    0.409716       0.465447     0.124837
Saved station-level ratio check to: Signal_output_split/signal_station_ratio_check_split.csv


In [24]:
# Preview daily turn ratios
daily_ratio[[
    "SignalID", "date",
    "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]].head()

,SignalID,date,intersection_type,left_ratio,through_ratio,right_ratio
0,75,2025-05-01,2_way,0.459771,0.330069,0.210160
1,75,2025-05-02,2_way,0.474103,0.323390,0.202508
2,75,2025-05-03,2_way,0.455439,0.331050,0.213512
3,75,2025-05-04,2_way,0.488306,0.312806,0.198888
4,75,2025-05-05,2_way,0.478571,0.323506,0.197922


In [25]:
# Preview weekday-average turn ratios
weekday_ratio[[
    "SignalID", "dow", "dow_name",
    "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]].head()

,SignalID,dow,dow_name,intersection_type,left_ratio,through_ratio,right_ratio
0,75,0.0,Monday,2_way,0.470862,0.329506,0.199631
1,75,1.0,Tuesday,2_way,0.464495,0.329564,0.205941
2,75,2.0,Wednesday,2_way,0.466400,0.327196,0.206405
3,75,3.0,Thursday,2_way,0.465443,0.328514,0.206043
4,75,4.0,Friday,2_way,0.463011,0.328286,0.208703


In [26]:
# Daily L/T/R ratios by intersection type
ratio_cols = ["left_ratio", "through_ratio", "right_ratio"]

daily_type_daily = (
    daily_ratio
    .groupby(["intersection_type", "date"])[ratio_cols]
    .mean()
    .reset_index()
)

daily_type_daily.head()

,intersection_type,date,left_ratio,through_ratio,right_ratio
0,2_way,2025-05-01,0.296943,0.610967,0.092090
1,2_way,2025-05-02,0.301536,0.608032,0.090432
2,2_way,2025-05-03,0.298478,0.605273,0.096248
3,2_way,2025-05-04,0.301644,0.604466,0.093890
4,2_way,2025-05-05,0.300222,0.609136,0.090642


In [27]:
# Weekday L/T/R ratios by intersection type
weekday_type_daily = (
    weekday_ratio
    .groupby(["intersection_type", "dow", "dow_name"])[ratio_cols]
    .mean()
    .reset_index()
)

weekday_type_daily.head()

,intersection_type,dow,dow_name,left_ratio,through_ratio,right_ratio
0,2_way,0.0,Monday,0.300371,0.609264,0.090364
1,2_way,1.0,Tuesday,0.301422,0.608874,0.089704
2,2_way,2.0,Wednesday,0.301331,0.608586,0.090083
3,2_way,3.0,Thursday,0.301056,0.609656,0.089288
4,2_way,4.0,Friday,0.301044,0.608032,0.090924


In [28]:
# Overall L/T/R ratios by intersection type
overall_type_summary = (
    daily_ratio  
    .groupby("intersection_type")[ratio_cols]
    .mean()
    .reset_index()
)

overall_type_summary

,intersection_type,left_ratio,through_ratio,right_ratio
0,2_way,0.299286,0.608511,0.092203
1,4_way,0.174992,0.756350,0.068658
2,T_intersection,0.166140,0.760105,0.073755
3,other,0.749658,0.250342,0.000000


In [29]:
output_daily_by_date = os.path.join(DATA_FOLDER, "intersection_turn_ratios_daily_split.csv")
daily_type_daily.to_csv(output_daily_by_date, index=False)
print("Saved:", output_daily_by_date)

output_weekday_by_dow = os.path.join(DATA_FOLDER, "intersection_turn_ratios_weekday_split.csv")
weekday_type_daily.to_csv(output_weekday_by_dow, index=False)
print("Saved:", output_weekday_by_dow)

output_overall = os.path.join(DATA_FOLDER, "intersection_turn_ratios_overall_split.csv")
overall_type_summary.to_csv(output_overall, index=False)
print("Saved:", output_overall)


Saved: Signal_output_split/intersection_turn_ratios_daily_split.csv
Saved: Signal_output_split/intersection_turn_ratios_weekday_split.csv
Saved: Signal_output_split/intersection_turn_ratios_overall_split.csv
